# 02 — DA-WI Risk Relationships

Frozen evidence layer for the synthetic-data pipeline.

- DA-WI: India-specific risk-factor relationships (RRRs) and published weights.
- NFHS-5: population-level prevalence/reference values where available in our CSV.
- No synthetic generation or ML training is performed here.

Source: Sabri et al. (2024), Table 3, DA-WI. The study used longitudinal data from 150 women in India; the weighted DA-WI reported AUC 0.803 for future severe IPV.

In [ ]:
import pandas as pd
import numpy as np
import os

nfhs_violence=pd.read_csv('../data/processed/nfhs_violence_reference.csv',encoding='latin1')
nfhs_relevant=pd.read_csv('../data/processed/nfhs_relevant_indicators.csv',encoding='latin1')
print('NFHS violence reference:',nfhs_violence.shape)
print('NFHS relevant indicators:',nfhs_relevant.shape)

## 1. Published DA-WI factors

The published RRR-to-weight rule is: <1.33→0, 1.33–1.79→1, 1.80–2.79→2, 2.80–3.79→3, ≥3.80→4. Strangulation was intentionally increased from 2 to 3.

In [ ]:
dawi=pd.DataFrame(dawi_rows,columns=['feature','dawi_item','rrr','weight','source_group'])
print('DA-WI factors:',len(dawi))
display(dawi)

In [ ]:
def rrr_to_weight(rrr):
    if rrr<1.33:return 0
    if rrr<=1.79:return 1
    if rrr<=2.79:return 2
    if rrr<=3.79:return 3
    return 4

dawi['rule_weight']=dawi['rrr'].apply(rrr_to_weight)
dawi['weight_matches_rule']=dawi['rule_weight'].eq(dawi['weight'])
display(dawi[['feature','rrr','weight','rule_weight','weight_matches_rule']])
print('Rule matches all published weights:',dawi['weight_matches_rule'].all())
print('Note: strangulation is the intentional published override.')

## 2. NFHS-5 → DA-WI mapping

The downloaded NFHS CSV is aggregated by state/UT. We only mark a factor as directly supported when the file actually contains that concept. We do not invent missing NFHS measurements.

In [ ]:
mapping=pd.DataFrame(mapping_rows,columns=['feature','nfhs_status','mapping_note'])
dawi_mapping=dawi.merge(mapping,on='feature',how='left')
display(dawi_mapping)

In [ ]:
spousal=nfhs_violence['sub indicators'].astype(str).str.contains('spousal violence',case=False,na=False)
pregnancy=nfhs_violence['sub indicators'].astype(str).str.contains('physical violence during any pregnancy',case=False,na=False)
sexual=nfhs_violence['sub indicators'].astype(str).str.contains('sexual violence',case=False,na=False)
print('Spousal violence state rows:',spousal.sum())
print('Pregnancy violence state rows:',pregnancy.sum())
print('Sexual violence state rows:',sexual.sum())
display(nfhs_violence.loc[spousal|pregnancy|sexual].head(20))

## 3. DA-WI weighted reference score

This reproduces the published weighted DA-WI structure. It is a reference score, **not** the final target label for our app model.

In [ ]:
DAWI_WEIGHTS=dict(zip(dawi['feature'],dawi['weight']))
def calculate_dawi_score(row):
    return sum(int(row.get(feature,0))*weight for feature,weight in DAWI_WEIGHTS.items())
print('Number of factors:',len(DAWI_WEIGHTS))
print('Maximum weight sum from this table:',sum(DAWI_WEIGHTS.values()))
print('Published DA-WI weighted score range: 0–64')

## 4. Save outputs for Notebook 03

In [ ]:
os.makedirs('../data/processed',exist_ok=True)
dawi.to_csv('../data/processed/dawi_risk_factors.csv',index=False)
dawi_mapping.to_csv('../data/processed/dawi_nfhs_mapping.csv',index=False)
print('Saved dawi_risk_factors.csv and dawi_nfhs_mapping.csv')

## Frozen output

Notebook 03 will use the 26 DA-WI factors, published RRRs/weights, and the NFHS prevalence references. App-specific immediate-safety variables remain separate from DA-WI factors.